<!-- cspell:ignore atleast dalitz mystnb -->

# Phase space backends

:::{autolink-concat}
:::

{class}`.TFPhaseSpaceGenerator` and {class}`.TFWeightedPhaseSpaceGenerator` generate phase space samples with [`phasespace`](https://phasespace.readthedocs.io), which can run on NumPy, TensorFlow, or JAX. Despite their names, these generators use NumPy by default and do not require TensorFlow. This page compares the backends, so that you can decide which one fits your use case. All of them return a {obj}`.DataSample` of NumPy arrays with the same structure, so they can be swapped without changing the rest of an analysis.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib_inline.backend_inline import set_matplotlib_formats

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
set_matplotlib_formats("svg")

## Selecting a backend

The backend is selected with the `backend` argument of the generator. The mapping below is the single place that defines which backends this page compares. All tables and figures on this page loop over it.

In [ ]:
from tensorwaves.data import (
    NumpyUniformRNG,
    TFPhaseSpaceGenerator,
    TFUniformRealNumberGenerator,
)

initial_state_mass = 3.0969
final_state_masses = {0: 0.1350, 1: 0.1396, 2: 0.4937}
generators = {
    "numpy": (
        TFPhaseSpaceGenerator(initial_state_mass, final_state_masses, backend="numpy"),
        NumpyUniformRNG(seed=0),
    ),
    "tensorflow": (
        TFPhaseSpaceGenerator(
            initial_state_mass, final_state_masses, backend="tensorflow"
        ),
        TFUniformRealNumberGenerator(seed=0),
    ),
    "jax": (
        TFPhaseSpaceGenerator(initial_state_mass, final_state_masses, backend="jax"),
        NumpyUniformRNG(seed=0),
    ),
}
for generator, _ in generators.values():
    generator.show_progress = False

| `backend` | Extra | Use of the {class}`.RealNumberGenerator` |
| --- | --- | --- |
| `"numpy"` (default) | `phsp` | {class}`.NumpyUniformRNG` is used directly, any other type only provides a seed |
| `"tensorflow"` | `phsp,tf` | {class}`.TFUniformRealNumberGenerator` is used directly, any other type only provides a seed |
| `"jax"` | `phsp,jax` | any type only provides a seed |

The installation of these extras is described on {doc}`/install`.

## Generating a sample

Every backend is used through the same {meth}`~.TFPhaseSpaceGenerator.generate` method. TensorFlow and JAX compile functions the first time they are called, so we generate two samples per backend and record the time of each. These timings only give an impression for this three-body decay on the machine that builds this page.

In [ ]:
n_events = 100_000
samples = {}
timings = {}
for name, (generator, rng) in generators.items():
    start = time.perf_counter()
    samples[name] = generator.generate(n_events, rng)
    first_call = time.perf_counter() - start
    start = time.perf_counter()
    generator.generate(n_events, rng)
    timings[name] = first_call, time.perf_counter() - start

In [ ]:
pd.DataFrame(
    {
        "Array type": [type(sample["p0"]).__name__ for sample in samples.values()],
        "Shape": [sample["p0"].shape for sample in samples.values()],
        "First call [s]": [round(first, 3) for first, _ in timings.values()],
        "Second call [s]": [round(second, 3) for _, second in timings.values()],
    },
    index=list(samples),
)

## Comparing distributions

The events of different backends are not identical, not even with the same seed, but they should follow the same distribution. A uniform phase space sample shows up as a flat Dalitz plot within the kinematic boundary, and the invariant mass projections should agree between backends.

In [ ]:
def invariant_mass_squared(p: np.ndarray) -> np.ndarray:
    return p[:, 0] ** 2 - np.sum(p[:, 1:] ** 2, axis=1)


fig, axes = plt.subplots(
    ncols=len(samples), figsize=(4 * len(samples), 3.5), sharex=True, sharey=True
)
for ax, (name, sample) in zip(np.atleast_1d(axes), samples.items(), strict=True):
    s01 = invariant_mass_squared(sample["p0"] + sample["p1"])
    s12 = invariant_mass_squared(sample["p1"] + sample["p2"])
    ax.hist2d(s01, s12, bins=60, cmin=1, rasterized=True)
    ax.set_title(name)
    ax.set_xlabel("$m_{01}^2$ [GeV$^2$]")
np.atleast_1d(axes)[0].set_ylabel("$m_{12}^2$ [GeV$^2$]")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for name, sample in samples.items():
    m01 = np.sqrt(invariant_mass_squared(sample["p0"] + sample["p1"]))
    ax.hist(m01, bins=60, histtype="step", label=name)
ax.set_xlabel("$m_{01}$ [GeV]")
ax.set_ylabel("Events")
ax.legend()
fig.tight_layout()
plt.show()

## Which backend to choose

NumPy is the default choice, because it needs no additional dependencies and no compilation. TensorFlow reproduces samples from earlier TensorWaves versions, when combined with a {class}`.TFUniformRealNumberGenerator`. JAX can be useful if the rest of the analysis already uses it.

There are two things to keep in mind:

- The same seed gives a different sample on each backend. Samples are only reproducible within one backend.
- `phasespace` selects its backend per process, and a generator activates its backend each time it generates. Each switch makes `phasespace` recompile, so alternating between backends is slower than the timings above suggest, and generators with different backends should not be used concurrently in threads.